# FedIT-FL 运行 notebook

联邦指令微调（Qwen2.5-0.5B + LoRA，Dolly-15k）上的 CSDD free rider 检测。每一格都可以单独运行（各格开头都会先进入项目目录），不依赖前面的格子，重启内核后也不用从头跑。

- 实验在后台运行，关掉网页或重启内核都不会中断
- 停止后重新运行“启动”那一格即可从断点续跑，已完成的小实验会跳过
- 服务器日志里的时间是北京时间

顺序：0 解压 → 1 下载模型 → 2 环境检查 → 3 学习率探测 → 4 试点 → 6 看结果 → 7 正式实验

## 0. 首次使用：解压（已经解压过就跳过）

解压前先关掉本项目里打开的 notebook 标签页（不保存），实验运行中不要重新解压。

In [ ]:
%cd /root/autodl-tmp
!python3 -m zipfile -e FedIT-FL.zip .

## 1. 下载基座模型 Qwen2.5-0.5B（只需一次，约 1 GB）

依次尝试 ModelScope（国内快）、hf-mirror、huggingface.co，断了重新运行会续传。最后一行显示 `ok: model_cache/Qwen2.5-0.5B (qwen2, 24 layers, ...)` 就是成功。

如果服务器下载太慢：在 Mac 上下载 Qwen/Qwen2.5-0.5B 的 `config.json`、`generation_config.json`、`model.safetensors`、`tokenizer.json`、`tokenizer_config.json`、`vocab.json`、`merges.txt`，拖进 JupyterLab 的 `FedIT-FL/model_cache/Qwen2.5-0.5B/` 文件夹，再运行这一格检查。

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
!bash start.sh fetch-model

## 2. 状态与环境检查（第一次会把依赖装到 pip_pkgs/，不动平台自带的 PyTorch）

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
!bash start.sh status
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv
!PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple bash run.sh --help > /dev/null && echo "环境 OK"

## 3. 学习率探测（后台运行）

无攻击、α=0.5、不算 Shapley，同时跑 5e-5 / 1e-4 / 2e-4 / 5e-4 四个学习率，每个 25 轮。进度用第 5 节查看。跑完后运行下一格：表里第 -1 行是未训练的基座模型；脚本会建议一个学习率（最后几轮仍在下降、且第 0 轮占总下降不到一半），把它填到第 4、6、7 节的 `LR`。

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
!PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple bash start.sh lr-probe

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
!PYTHONPATH=/root/autodl-tmp/FedIT-FL/pip_pkgs python3 lr_probe_report.py results/lr_probe
from IPython.display import Image, display
import os
if os.path.exists("results/lr_probe/lr_probe.png"):
    display(Image("results/lr_probe/lr_probe.png"))

## 4. 试点（8 个小实验，后台运行）

seed 42，IID 和 α=0.1 ×（无攻击 + DFR / SDFR / AFR 30%）。每次训练同时算 6 种分组的 Shapley：LLM 知识单元（主组）、同样大小的随机单元、Dolly 原生 8 类、随机 8 组、TF-IDF 聚类，以及不分类别的全局 Shapley。

看到 `started pilot (pid …)` 就是启动成功。第一轮跑完后用第 5 节看每轮耗时（日志里的 `total=…s`）。学习率不是 1e-4 时结果写到 `results/pilot_lr<LR>`。

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
LR = "1e-4"   # 学习率探测后改成选定的值
!LOCAL_LR={LR} PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple bash start.sh pilot

## 5. 查看进度（随时运行）

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
!bash start.sh status

## 6. 查看结果（运行中也可以看已完成部分）

先把已完成的小实验合并成总表 `fedit_csdd_summary.xlsx`，再显示各分组的 AUC 对比、LLM 单元对各对照的配对检验、AUC 表、测试损失和总图。`ROOT` 选试点或正式实验。

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
import glob
from IPython.display import Image, display
LR = "1e-4"        # 和第 4 / 7 节相同
STAGE = "pilot"    # 或 "full"
ROOT = f"results/{STAGE}" + ("" if LR in ("1e-4", "1e-04", "0.0001") else f"_lr{LR}")
!PYTHONPATH=/root/autodl-tmp/FedIT-FL/pip_pkgs python3 summarize.py --results-root {ROOT} > /dev/null
!PYTHONPATH=/root/autodl-tmp/FedIT-FL/pip_pkgs python3 show_summary.py {ROOT}
for p in sorted(glob.glob(f"{ROOT}/plots/*.png")):
    print(p); display(Image(p))

## 7. 正式实验：120 个小实验（后台运行）

4 种划分（IID、α=0.1/0.5/1.0）×（无攻击 + 3 种攻击 × 10/30/50%）× 3 个种子，按划分依次运行。种子改成 5 个：在 bash 前加 `SEEDS=42,43,44,45,46`。

两台机器分摊：一台 `TASKS="iid dirichlet:0.1"`，另一台 `TASKS="dirichlet:0.5 dirichlet:1.0"`，跑完把两边的 `results/full*` 合到一个文件夹再运行第 6 节。

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
LR = "1e-4"   # 学习率探测后改成选定的值
!LOCAL_LR={LR} PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple bash start.sh full

## 8. 查看某一个小实验的 7 张图

先运行一次列出所有已完成的小实验，再把 `PICK` 改成想看的编号。

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
import glob, os
from IPython.display import Image, display
runs = sorted(os.path.dirname(p) for p in glob.glob("results/**/runs/*/summary.json", recursive=True))
for i, r in enumerate(runs):
    print(i, r)
PICK = -1   # 改成编号；-1 = 最后一个
if runs:
    for p in sorted(glob.glob(os.path.join(runs[PICK], "plots", "*.png"))):
        print(p); display(Image(p))

## 9. 打包结果下载

打包后在左侧文件栏进入 `autodl-tmp/FedIT-FL`，右键 zip 选 Download。

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
import time
name = f"fedit_results_{time.strftime('%Y%m%d_%H%M')}.zip"
!python3 -m zipfile -c {name} results logs
!ls -lh {name}

## 10. 停止

停止后重新运行对应的启动格即可续跑。

In [ ]:
%cd /root/autodl-tmp/FedIT-FL
!bash start.sh stop